### STEP 1 : IMPORT EVERYTHING WE NEED

In [29]:
import os
from dotenv import load_dotenv

# langchain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings
from langchain_community.vectorstores import FAISS

In [30]:
load_dotenv()

True

In [31]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

print("ENV VAR LOADED")

ENV VAR LOADED


### STEP 2 : LOADING OUR DATA

In [32]:
DATA_FILE_PATH = os.path.join("data", "hr_policy.txt")

### STEP 3 : DATA INGESTION

In [33]:
loader = TextLoader(DATA_FILE_PATH , encoding="utf-8")

documents = loader.load()

print("DATA LOADED")
print("="*40)
print(documents)

DATA LOADED
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDu

### LANGCHAIN DOCUMENT
lanchain processes everything in form of documents.

document : 

1. page content => actual data
2. meta data => extra information about the data

In [34]:
len(documents)

1

In [35]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [36]:
print(documents[0].metadata)

{'source': 'data\\hr_policy.txt'}


### STEP 4 : SPLITTING OUR DATA

In [37]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

chunks = text_splitter.split_documents(documents)
print(chunks)
# now each splited chunk is a document and page content and meta data

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

In [38]:
print(chunks[5].page_content)

5. REIMBURSEMENT POLICY
Employees can claim reimbursement for approved business expenses such as travel,
client meals, and internet bills used for official work.
All reimbursement claims must be submitted with valid bills within 30 days of the expense.
Claims are processed within 10 working days after approval from the reporting manager.


### STEP 5 : EMBEDDING

In [39]:
from langchain_community.embeddings import JinaEmbeddings

embedding_model = JinaEmbeddings(model_name="Jina-embedding-v2-base-en")
print("EMB MODEL READY THE NAME IS ", embedding_model.model_name)

EMB MODEL READY THE NAME IS  Jina-embedding-v2-base-en


In [40]:
from langchain_community.embeddings import JinaEmbeddings

embedding_model = JinaEmbeddings(
    jina_api_key="jina_1a828e3ba8d84b9a8331948779c41dddga2oYGRaYp2UZPvl0OrTVMLy1WeI",
    model_name="jina-embeddings-v2-base-en"
)

### STEP 6 : STORE VECTOR IN DB

In [41]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks, embedding_model)

print("CHUNKS ARE STORED:", vector_store.index.ntotal)

CHUNKS ARE STORED: 9


### SIMILARITY SEARCH TECHNIQUE

In [42]:
test_query = "how many sick leaves employees get"
## similarity search
top_matches = vector_store.similarity_search(test_query, k = 2)
print(f"query: {test_query}\n")
for i, match in enumerate(top_matches,start=1):
    print(f"--- Match{i} ---")
    print(match.page_content)
    print()

query: how many sick leaves employees get

--- Match1 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--- Match2 ---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



### TOOL

In [43]:
retriever = vector_store.as_retriever(search_kwargs={"k" : 3}) # return top 3 relevant answer
def search_hr_policy(question:str) -> str:
    """
    Search the hr policy document for information about leave, work from home, 
    probation, notice period, reimburstment, code of conduct, holidays, or exit process
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunks.page_content for chunks in matching_chunks)

### DATA RETRIVAL PIPELINE

In [44]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model= "openai/gpt-oss-120b" ,
    temperature= 0.9 # tells you bout creativity
)
llm.model_name


'openai/gpt-oss-120b'

### INVOKE METHOD

In [45]:
test_response = llm.invoke("hey is learning rag hard ? answer in 1 line")

In [46]:
test_response.content

'Learning RAG can be challenging at first, but with solid fundamentals in retrieval and generation, it becomes manageable.'

### MAKING AN AI AGENT

AI AGENT contains 3 things =>
1. llm = brain
2. tool = super power
3. memory = no memory

In [52]:

from langchain.agents import create_agent

hr_assistent = create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt="""
    You are a friendly HR assistant.
    Always use the search_hr_policy tool to look up facts before answering.
    If the answer isn't in the search result, say you don't know.
    Do not guess.
    """
)

print("HR assistant is ready to answer the questions!")


HR assistant is ready to answer the questions!


In [53]:

def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""

    print("=" * 60)
    print("Question:", question)
    print("-" * 60)

    response = hr_assistent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question
                }
            ]
        }
    )

    answer = response["messages"][-1].content

    print("Answer:", answer)
    print("=" * 60)
    print()

    return answer


In [54]:

response = hr_assistent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Tell me about the leave policies."
            }
        ]
    }
)

response


{'messages': [HumanMessage(content='Tell me about the leave policies.', additional_kwargs={}, response_metadata={}, id='5bdbe430-cf95-4426-95d5-d1f7424b24a3'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "Tell me about the leave policies." According to developer instructions, we must always use search_hr_policy tool to look up facts before answering. So we need to call search_hr_policy with question about leave policies. Then based on result, answer. If not found, say we don\'t know. Let\'s do that.', 'tool_calls': [{'id': 'fc_59758a4a-d224-404a-9f62-fc317de9f3be', 'function': {'arguments': '{"question":"leave policies"}', 'name': 'search_hr_policy'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 203, 'total_tokens': 299, 'completion_time': 0.199934109, 'completion_tokens_details': {'reasoning_tokens': 67}, 'prompt_time': 0.007744785, 'prompt_tokens_details': None, 'queue_time': 0.211471602, 'tota

In [55]:
response["messages"][-1].content

'Here’s a concise overview of our company’s leave policies, based on the current HR policy document:\n\n| Type of Leave | Entitlement | Key Rules |\n|---------------|-------------|-----------|\n| **Annual (Paid) Leave** | **20\u202fdays per calendar year** for full‑time employees | • Requests must be entered in the HR portal **at least 5\u202fworking days before the intended start date**.<br>• Up to **5\u202fdays** of unused annual leave can be carried forward to the next year; any remaining balance beyond that is forfeited. |\n| **Sick Leave** | **10\u202fpaid days per year** | • Treated separately from annual leave.<br>• A **medical certificate** is required for any sick‑leave period that exceeds **2 consecutive days**. |\n| **Public Holidays** | **12\u202fdays** per year (as published in the annual holiday calendar) | • If you work on a public holiday, you are eligible for **compensatory leave**. |\n| **Leave Request Process** | – | • All leave requests (annual, sick, or compensator

In [56]:
ask_hr_assistant("What is the HR policy?")

Question: What is the HR policy?
------------------------------------------------------------
Answer: Here’s a quick overview of the HR policy sections that are currently available in the handbook:

**1. Leave Policy**  
- **Annual leave:** 20 paid days per calendar year. Requests must be submitted through the HR portal at least 5 working days in advance.  
- **Carry‑forward:** Up to 5 unused days can be carried over to the next year.  
- **Sick leave:** 10 paid sick days per year. A medical certificate is required for sick leave longer than 2 consecutive days.

**2. Exit Policy**  
- **Clearance:** When you resign or are terminated, you must complete a clearance process with IT, Finance, and HR before your last working day.  
- **Final settlement:** Full and final settlement—including any pending reimbursements and leave encashment—is processed within 45 days of your last working day.

These are the HR policy details that are currently documented. If you need information on other topi

'Here’s a quick overview of the HR policy sections that are currently available in the handbook:\n\n**1. Leave Policy**  \n- **Annual leave:** 20\u202fpaid days per calendar year. Requests must be submitted through the HR portal at least\u202f5\u202fworking days in advance.  \n- **Carry‑forward:** Up to\u202f5\u202funused days can be carried over to the next year.  \n- **Sick leave:** 10\u202fpaid sick days per year. A medical certificate is required for sick leave longer than\u202f2\u202fconsecutive days.\n\n**2. Exit Policy**  \n- **Clearance:** When you resign or are terminated, you must complete a clearance process with IT, Finance, and HR before your last working day.  \n- **Final settlement:** Full and final settlement—including any pending reimbursements and leave encashment—is processed within\u202f45\u202fdays of your last working day.\n\nThese are the HR policy details that are currently documented. If you need information on other topics (e.g., work‑from‑home, probation, not